In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import itertools, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox

# Configuration
ROOT = os.environ.get("AGL_ROOT", ".")


def p(*parts):
    return os.path.join(ROOT, *parts)


LEVELS_FILE = p("Code Outputs", "Gap Interpolation Outputs",
                "Unified_Interpolated_Levels.xlsx")
OUT_DIR = p("Code Outputs", "Baseline Outputs")
ARIMA_DIR = p("Code Outputs", "Arima Forecast Outputs")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ARIMA_DIR, exist_ok=True)

START, END = "1995-06-01", "2025-12-01"   # canonical common window
TEST_MONTHS = 60
HORIZONS = [1, 3, 6, 12]
D_ORDER = 1          # levels are I(1) (ADF/KPSS, EDA step 3)
S = 12               # seasonal period
PQ_MAX = 3           # search p,q in 0..3
MAXITER = 1000
METHOD = "lbfgs"
EXCLUDE_INTERP = True


def rmse(pred, act):
    pred, act = np.asarray(pred, float), np.asarray(act, float)
    return float(np.sqrt(np.nanmean((pred - act) ** 2)))


def mae(pred, act):
    pred, act = np.asarray(pred, float), np.asarray(act, float)
    return float(np.nanmean(np.abs(pred - act)))


def fit_sarimax(y, order, seasonal_order):
    """Single canonical fit call - every SARIMA/ARIMA fit in the project uses this."""
    return SARIMAX(y, order=order, seasonal_order=seasonal_order,
                   enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False, method=METHOD, maxiter=MAXITER)


# Loading
df = pd.read_excel(LEVELS_FILE)
df["Date"] = pd.to_datetime(df["Date"])

L = (df.pivot(index="Date", columns="Reservoir", values="Level_m")
       .sort_index().asfreq("MS").loc[START:END])
IN = (df.pivot(index="Date", columns="Reservoir", values="is_interpolated")
        .reindex(L.index).fillna(False).astype(bool))

assert L.notna().all().all(), "NaNs inside the common window - check START/END"
LAKES = list(L.columns)
N = len(L)
split = N - TEST_MONTHS
origins = range(split, N)

print("=" * 76)
print("SHARED BASELINE - canonical protocol")
print("=" * 76)
print(f"  window        : {L.index[0].date()} .. {L.index[-1].date()}  ({N} months)")
print(f"  training      : {L.index[0].date()} .. {L.index[split-1].date()}  ({split} months)")
print(f"  test period   : {L.index[split].date()} .. {L.index[-1].date()}  ({TEST_MONTHS} months)")
print(f"  lakes         : {len(LAKES)}  {LAKES}")
print(f"  interpolated months inside the test period: "
      f"{int(IN.iloc[split:].to_numpy().sum())}")
print()


# Order selection
def select_arima(train):
    best_order, best_aic = None, np.inf
    for pp, qq in itertools.product(range(PQ_MAX + 1), range(PQ_MAX + 1)):
        try:
            aic = fit_sarimax(train, (pp, D_ORDER, qq), (0, 0, 0, 0)).aic
            if np.isfinite(aic) and aic < best_aic:
                best_order, best_aic = (pp, D_ORDER, qq), aic
        except Exception:
            pass
    return best_order, best_aic


def select_seasonal(train, pdq):
    best_order, best_aic = None, np.inf
    for P, Dn, Q in itertools.product([0, 1], [0, 1], [0, 1]):
        if P == 0 and Dn == 0 and Q == 0:
            continue
        try:
            aic = fit_sarimax(train, pdq, (P, Dn, Q, S)).aic
            if np.isfinite(aic) and aic < best_aic:
                best_order, best_aic = (P, Dn, Q, S), aic
        except Exception:
            pass
    return best_order, best_aic


# Rolling-origin forecasts
def rolling_stat(series, order, seasonal_order):
    """Fit once on train, then filter forward one month at a time.
    Returns {(target_idx, horizon): (origin_idx, prediction)} and the fit object."""
    res = fit_sarimax(series.iloc[:split], order, seasonal_order)
    converged = bool(res.mle_retvals.get("converged", False))
    store, walker = {}, res
    for o in origins:
        steps = min(max(HORIZONS), N - o)
        fc = walker.get_forecast(steps=steps).predicted_mean.values
        for h in HORIZONS:
            if h <= steps:
                store[(o + h - 1, h)] = (o, fc[h - 1])
        walker = walker.append(series.iloc[o:o + 1], refit=False)
    return store, res, converged


def rolling_naive(series, kind):
    store, v = {}, series.values
    for o in origins:
        for h in HORIZONS:
            tgt = o + h - 1
            if tgt >= N:
                continue
            if kind == "rw":
                store[(tgt, h)] = (o, v[o - 1])          # last observed value
            else:                                        # seasonal naive
                src = tgt - S
                store[(tgt, h)] = (o, v[src] if src >= 0 else np.nan)
    return store


# Run per lake
pred_rows, order_rows, metric_rows = [], [], []
t0 = time.time()

for lk in LAKES:
    s = L[lk]
    train = s.iloc[:split]
    interp_flag = IN[lk].to_numpy()

    arima_order, arima_aic = select_arima(train)
    seas_order, seas_aic = select_seasonal(train, arima_order)

    diag = fit_sarimax(train, arima_order, seas_order)
    lb_p = float(acorr_ljungbox(diag.resid[S:], lags=[12],
                                return_df=True)["lb_pvalue"].iloc[0])

    stores = {
        "RandomWalk":    rolling_naive(s, "rw"),
        "SeasonalNaive": rolling_naive(s, "snaive"),
    }
    a_store, a_res, a_conv = rolling_stat(s, arima_order, (0, 0, 0, 0))
    sa_store, sa_res, sa_conv = rolling_stat(s, arima_order, seas_order)
    stores["ARIMA"] = a_store
    stores["SARIMA"] = sa_store

    print(f"  {lk:16s} ARIMA{arima_order} (AIC {arima_aic:8.1f}) | "
          f"SARIMA{arima_order}x{seas_order} (AIC {seas_aic:8.1f}) | "
          f"LB p={lb_p:.3f} | converged: ARIMA={a_conv} SARIMA={sa_conv}")

    order_rows.append({
        "Lake": lk,
        "ARIMA_order": str(arima_order),
        "SARIMA_seasonal": str(seas_order),
        "ARIMA_AIC": round(arima_aic, 1),
        "SARIMA_AIC": round(seas_aic, 1),
        "LjungBox_p_lag12": round(lb_p, 3),
        "residuals_white": "yes" if lb_p > 0.05 else "no",
        "ARIMA_converged": a_conv,
        "SARIMA_converged": sa_conv,
    })

    # predictions + scoring
    for model, store in stores.items():
        by_h = {h: ([], []) for h in HORIZONS}
        for (tgt, h), (o, pred) in sorted(store.items()):
            is_int = bool(interp_flag[tgt])
            pred_rows.append({
                "Lake": lk, "Model": model,
                "Origin_Date": L.index[o], "Target_Date": L.index[tgt],
                "Horizon_m": h,
                "Pred_m": (None if pred is None or not np.isfinite(pred)
                           else round(float(pred), 6)),
                "Actual_m": round(float(s.iloc[tgt]), 6),
                "is_interpolated_target": is_int,
            })
            if EXCLUDE_INTERP and is_int:
                continue
            if pred is None or not np.isfinite(pred):
                continue
            by_h[h][0].append(pred)
            by_h[h][1].append(s.iloc[tgt])
        for h in HORIZONS:
            P, A = by_h[h]
            metric_rows.append({
                "Lake": lk, "Model": model, "Horizon_m": h,
                "RMSE_m": round(rmse(P, A), 4),
                "MAE_m": round(mae(P, A), 4),
                "n_scored": len(P),
            })

print(f"\n  ({time.time() - t0:.0f}s)")

# Skill columns and save
metrics = pd.DataFrame(metric_rows)
ref_rw = metrics[metrics.Model == "RandomWalk"].set_index(["Lake", "Horizon_m"])["RMSE_m"]
ref_sa = metrics[metrics.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]

metrics["skill_vs_RW_%"] = metrics.apply(
    lambda r: round(100 * (ref_rw[(r.Lake, r.Horizon_m)] - r.RMSE_m)
                    / ref_rw[(r.Lake, r.Horizon_m)], 1), axis=1)
metrics["skill_vs_SARIMA_%"] = metrics.apply(
    lambda r: round(100 * (ref_sa[(r.Lake, r.Horizon_m)] - r.RMSE_m)
                    / ref_sa[(r.Lake, r.Horizon_m)], 1), axis=1)

preds = pd.DataFrame(pred_rows)
orders = pd.DataFrame(order_rows)

preds.to_csv(os.path.join(OUT_DIR, "Baseline_predictions.csv"), index=False)
metrics.to_csv(os.path.join(OUT_DIR, "Baseline_metrics.csv"), index=False)
orders.to_csv(os.path.join(OUT_DIR, "Baseline_orders.csv"), index=False)

# Backwards-compatible copies so existing downstream paths keep working.
orders[["Lake", "ARIMA_order", "SARIMA_seasonal", "SARIMA_AIC",
        "LjungBox_p_lag12", "residuals_white"]].to_csv(
    os.path.join(ARIMA_DIR, "FC_orders.csv"), index=False)
metrics[["Lake", "Model", "Horizon_m", "RMSE_m", "MAE_m", "skill_vs_RW_%"]].to_csv(
    os.path.join(ARIMA_DIR, "FC_metrics.csv"), index=False)

# Report
print("\n=== CHOSEN ORDERS & DIAGNOSTICS (canonical window) ===")
print(orders.to_string(index=False))

print("\n=== RMSE (m) by model x horizon (mean over lakes) ===")
print(metrics.pivot_table(index="Model", columns="Horizon_m",
                          values="RMSE_m").round(4).to_string())

print("\n=== mean skill vs RandomWalk (%) ===")
print(metrics.pivot_table(index="Model", columns="Horizon_m",
                          values="skill_vs_RW_%").round(1).to_string())

print("\n=== forecasts scored per lake x horizon (should be 60/58/55/49) ===")
print(metrics[metrics.Model == "SARIMA"].pivot(
    index="Lake", columns="Horizon_m", values="n_scored").to_string())


# Figure - test-period actual vs 1-step-ahead SARIMA and RandomWalk
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

one = preds[preds.Horizon_m == 1]
fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()
for i, lk in enumerate(LAKES):
    ax = axes[i]
    ctx = L[lk].iloc[split - 24:]
    ax.plot(ctx.index, ctx.values, color="k", lw=1.2, label="actual")
    for model, col in [("SARIMA", "tab:red"), ("RandomWalk", "tab:blue")]:
        d = (one[(one.Lake == lk) & (one.Model == model)]
             .sort_values("Target_Date"))
        ax.plot(d["Target_Date"], d["Pred_m"], col, lw=1, alpha=.85,
                label=f"{model} 1-step")
    ax.axvline(L.index[split], color="grey", ls="--", lw=.9)
    ax.set_title(lk, fontweight="bold")
    ax.set_ylabel("Level (m)")
    if i == 0:
        ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Rolling-origin 1-step forecasts over the test period "
             "(dashed = train/test split)", fontweight="bold", y=.999)
plt.tight_layout()
for d in (OUT_DIR, ARIMA_DIR):
    plt.savefig(os.path.join(d, "FC_forecast_plots.png"), dpi=200)
plt.close()

print(f"\nDone. Wrote:\n  {OUT_DIR}/Baseline_predictions.csv"
      f"\n  {OUT_DIR}/Baseline_metrics.csv"
      f"\n  {OUT_DIR}/Baseline_orders.csv"
      f"\n  {ARIMA_DIR}/FC_orders.csv   (compatibility copy)"
      f"\n  {ARIMA_DIR}/FC_metrics.csv  (compatibility copy)"
      f"\n  {ARIMA_DIR}/FC_forecast_plots.png")

SHARED BASELINE - canonical protocol
  window        : 1995-06-01 .. 2025-12-01  (367 months)
  training      : 1995-06-01 .. 2020-12-01  (307 months)
  test period   : 2021-01-01 .. 2025-12-01  (60 months)
  lakes         : 7  ['Lake Albert', 'Lake Edward', 'Lake Kivu', 'Lake Malawi', 'Lake Tanganyika', 'Lake Turkana', 'Lake Victoria']
  interpolated months inside the test period: 0

  Lake Albert      ARIMA(3, 1, 3) (AIC   -304.3) | SARIMA(3, 1, 3)x(1, 0, 1, 12) (AIC   -324.3) | LB p=0.000 | converged: ARIMA=True SARIMA=True
  Lake Edward      ARIMA(2, 1, 3) (AIC   -491.9) | SARIMA(2, 1, 3)x(1, 0, 1, 12) (AIC   -481.6) | LB p=0.000 | converged: ARIMA=True SARIMA=True
  Lake Kivu        ARIMA(3, 1, 3) (AIC   -502.9) | SARIMA(3, 1, 3)x(1, 0, 1, 12) (AIC   -484.9) | LB p=0.000 | converged: ARIMA=True SARIMA=True
  Lake Malawi      ARIMA(3, 1, 2) (AIC   -450.5) | SARIMA(3, 1, 2)x(1, 0, 1, 12) (AIC   -509.9) | LB p=0.093 | converged: ARIMA=True SARIMA=True
  Lake Tanganyika  ARIMA(2, 1, 3